# Algoverse step-zero inference

TLDR: run the two protocol checkpoints separately and save inspectable smoke outputs. This notebook has not yet been executed on a GPU. Use an L4/A100 runtime; the specified BF16 setup is not supported by T4. No paid runtime is created automatically.

[Protocol](https://refusal-wording-safety-judgments.dharshanlab.chatgpt.site/) · [Setup and blockers](https://github.com/Dharshan2004/algoverse/blob/dharshan/inference-setup/SETUP.md)

Follow the research site: Ministral 3 8B is the first primary-model candidate; Gemma 4 12B is the second family. Screen both and apply the protocol gates. Model selection was confirmed by Priyadharshan on 11 October. Restart the runtime between models.

In [ ]:
!git clone --branch dharshan/inference-setup https://github.com/Dharshan2004/algoverse.git
%cd algoverse
!git rev-parse HEAD

## Install for initial smoke
Official packages only. Save the installed lock below; after a successful smoke, use that exact lock for later runs. If Colab requests a restart after installation, restart before continuing. The initial install is deliberately not described as an already tested environment.

In [ ]:
%pip install -U transformers accelerate bitsandbytes huggingface_hub sentencepiece mistral-common
!python -m pip freeze > /content/environment-lock.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
assert torch.cuda.is_bf16_supported(), "Use BF16-capable L4/A100 rather than T4"
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 2**30, "GiB")

If model access requires authentication, add HF_TOKEN via Colab Secrets. Never paste a token into this notebook or its outputs. Complete any model access agreement yourself before running.

In [ ]:
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    pass  # Public access may work without a token; gated access fails explicitly.

In [ ]:
!python -m unittest discover -s tests -v
!python inference.py --dry-run --variants B R N R2 --out /content/planned-inputs

## Run one model
Default: 10 synthetic examples × original/refusal × two mappings + 2 identity calls = 42 calls. These are plumbing checks, not benchmark results. The full variants are prepared above without model calls. Inspect GPU outputs before running another model. The runner will stop rather than truncate inputs or overwrite a previous directory.

In [ ]:
MODEL = "ministral"  # then run "gemma" in a fresh runtime
import subprocess
subprocess.run(["python", "inference.py", "--model", MODEL, "--out", f"/content/{MODEL}-smoke"], check=True)

In [ ]:
import json
from pathlib import Path
run_dir = Path(f"/content/{MODEL}-smoke")
rows = [json.loads(x) for x in (run_dir/"results.jsonl").read_text().splitlines()]
from collections import Counter
print("Rows:", len(rows), "Verdicts:", Counter(r["verdict"] for r in rows))
print("Peak GiB:", max(r["peak_gpu_bytes"] for r in rows)/2**30)
print("Identity:", (run_dir/"identity_check.json").read_text())
for row in rows[:4]:
    print(row["source_id"], row["variant"], row["mapping"], repr(row["raw_output"]), row["semantic_margin"])

## Save privately
Download the smoke artifacts and environment lock. Source response text and future gated data must not be committed to the public repository. Synthetic labels do not certify accuracy. The runner intentionally refuses final-test partitions. Dataset policy reconciliation, cross-model length checks and all research gates remain prerequisites for real experiments.

In [ ]:
import shutil
shutil.copy("/content/environment-lock.txt", run_dir/"environment-lock.txt")
archive = shutil.make_archive(str(run_dir), "zip", run_dir)
from google.colab import files
files.download(archive)